# RentSafe-MY — ELECTRA Bahasa, retrained (v2)

Second attempt at `mesolitica/electra-base-discriminator-bahasa-cased`. The v1 run
scored F1 0.9764 on validation and was still useless: shown anything outside the
synthetic training data it answered SCAM to almost everything, including plainly
legitimate messages, at ~0.8 confidence.

## What went wrong in v1

The F1 score hid it. These are the numbers that did not:

| symptom | v1 ELECTRA | XLM-R (same data) |
|---|---|---|
| p(scam) on 95 validation rows | **never below 0.1 or above 0.9** | 32 below 0.1, 59 above 0.9 |
| hand-written legitimate messages | **0 of 4 correct** | 3 of 4 correct |
| the one real legitimate conversation | **SCAM at 0.87** | LEGITIMATE at 0.15 |
| epochs 1–2 validation F1 | **0.7975 = the always-SCAM baseline** | learned immediately |
| training loss at the final epoch | 0.19, still falling | converged |

A model whose probabilities all sit between 0.1 and 0.9 has not learned a decision
boundary — it is hovering either side of 0.5, which happens to sort the synthetic
data and nothing else. Two epochs pinned exactly at the majority-class baseline says
the classification head had not started learning at all.

That is an optimisation failure, not a backbone failure. ELECTRA's discriminator
attaches a randomly initialised classification head, and it was given XLM-R's recipe:
2e-5 for 8 epochs. With 379 training rows at batch 16 that is 24 steps per epoch,
192 steps in total. XLM-R converged in that budget; ELECTRA did not.

## What this notebook changes

| | v1 | v2 | why |
|---|---|---|---|
| learning rate | 2e-5 | **5e-5** | ELECTRA fine-tunes near 1e-4 in the original paper; 2e-5 barely moved the fresh head |
| epochs | 8 | **20**, early stopping patience 5 | 192 steps was not enough; early stopping keeps it from overfitting once it converges |
| loss | unweighted | **class-weighted** | training data is 253 SCAM / 126 LEGITIMATE, and an undertrained model falls back on that 2:1 prior — exactly the observed failure |
| max length | 192 | **256** | ~9.5% of conversations were truncated, and the scam tell (deposit before viewing) usually lands late in the conversation |

Everything else — data, split, seed, batch size, metric — is unchanged.

**This is no longer a matched comparison against XLM-R.** It is an attempt to make
ELECTRA work. If it succeeds and you want the head-to-head for the report, XLM-R has
to be rerun on this recipe too.

## The bar it has to clear

Not F1. Section 7 checks the things F1 missed, and prints PASS or FAIL:

1. probabilities must spread past 0.1 and 0.9, not huddle around 0.5
2. hand-written legitimate messages must come back LEGITIMATE
3. the real (non-synthetic) validation conversations must be right

A v2 that scores 0.98 F1 and still fails those is the same broken model with a better
number attached.

**Before running: Runtime → Change runtime type → T4 GPU.**

## 1. GPU and dependencies

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

In [ ]:
!pip install -q "transformers==4.46.2" "accelerate>=0.26.0" scikit-learn

## 2. Upload the dataset

Pick `train.csv` and `val.csv` from `rentsafe-my/data/processed/random/` — the split
the currently deployed model uses. Run again with `grouped/` afterwards if v2 passes;
`grouped` is the honest generalisation number, `random` is the model you deploy
because it has seen all 20 template families rather than 15.

In [ ]:
import os
from google.colab import files

os.makedirs("data", exist_ok=True)
print("Select train.csv and val.csv from data/processed/random/ ...")

for name in files.upload():
    os.replace(name, os.path.join("data", name))

print("Uploaded:", sorted(os.listdir("data")))

## 3. Setup

`CLASS_WEIGHTS` is the one setting worth experimenting with if v2 still leans toward
SCAM. It is computed from the training data as `total / (2 * class_count)`, which
gives LEGITIMATE roughly 1.5 and SCAM roughly 0.75. Set `USE_CLASS_WEIGHTS = False` to
see what the higher learning rate alone achieves.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

MODEL = "mesolitica/electra-base-discriminator-bahasa-cased"
SPLIT_SET = "random"
FINAL_DIR = f"electra-malay-scam-{SPLIT_SET}-v2"

# The v2 changes.
LEARNING_RATE = 5e-5
EPOCHS = 20
MAX_LENGTH = 256
USE_CLASS_WEIGHTS = True
EARLY_STOPPING_PATIENCE = 5

SEED = 42
BATCH_SIZE = 16
LABELS = {"LEGITIMATE": 0, "SCAM": 1}
ID2LABEL = {0: "LEGITIMATE", 1: "SCAM"}

set_seed(SEED)


def load_frame(path):
    frame = pd.read_csv(path)
    frame["label"] = frame["label"].str.upper().replace({"LEGIT": "LEGITIMATE"})
    invalid = sorted(set(frame["label"]) - set(LABELS))
    if invalid:
        raise ValueError(f"Unsupported labels in {path}: {invalid}")
    return frame


class ScamDataset(torch.utils.data.Dataset):
    def __init__(self, frame, tokenizer):
        self.encodings = tokenizer(
            frame["text"].tolist(), truncation=True, max_length=MAX_LENGTH
        )
        self.labels = [LABELS[label] for label in frame["label"]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


def compute_metrics(prediction):
    predicted = np.argmax(prediction.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        prediction.label_ids, predicted, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(prediction.label_ids, predicted),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


tokenizer = AutoTokenizer.from_pretrained(MODEL)
train_frame = load_frame("data/train.csv")
val_frame = load_frame("data/val.csv")
train_dataset = ScamDataset(train_frame, tokenizer)
val_dataset = ScamDataset(val_frame, tokenizer)

counts = train_frame["label"].value_counts()
weights = torch.tensor(
    [len(train_frame) / (2 * counts[name]) for name in ("LEGITIMATE", "SCAM")],
    dtype=torch.float,
)
CLASS_WEIGHTS = weights if USE_CLASS_WEIGHTS else None

steps_per_epoch = int(np.ceil(len(train_dataset) / BATCH_SIZE))
lengths = np.array([len(tokenizer.tokenize(t)) for t in val_frame["text"]])

print(f"{len(train_frame)} train / {len(val_frame)} validation")
print("train labels:", dict(counts))
print(f"class weights: LEGITIMATE {weights[0]:.3f}, SCAM {weights[1]:.3f}"
      f"{'' if USE_CLASS_WEIGHTS else '  (disabled)'}")
print(f"{steps_per_epoch} steps/epoch x {EPOCHS} epochs = {steps_per_epoch * EPOCHS} "
      f"steps (v1 had {int(np.ceil(379 / 16)) * 8})")
print(f"truncated at MAX_LENGTH={MAX_LENGTH}: {(lengths > MAX_LENGTH).mean():.1%} "
      f"(was {(lengths > 192).mean():.1%} at 192)")

## 4. Train

`WeightedTrainer` is the only structural change — it swaps the default unweighted
cross-entropy for a class-weighted one. Everything else is stock `Trainer`.

Watch the first two epochs. In v1 they sat at F1 0.7975, the always-SCAM baseline. If
they do that again, the learning rate is still too low: stop, raise it to 1e-4, and
re-run this cell.

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer with a class-weighted cross-entropy loss.

    The default loss treats a missed scam and a false alarm as equally costly, which
    on 2:1 imbalanced data lets an undertrained model coast by answering SCAM.
    """

    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = (
            self.class_weights.to(outputs.logits.device)
            if self.class_weights is not None
            else None
        )
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label=ID2LABEL, label2id=LABELS,
)

training_args = TrainingArguments(
    output_dir="checkpoints-v2",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10,
    seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    class_weights=CLASS_WEIGHTS,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

trainer.train()

## 5. Did it converge this time?

The training curve, and the check v1 would have failed. A converged run shows loss
falling well below 0.19 and F1 leaving the 0.7975 baseline within the first epoch or
two.

In [ ]:
history = [e for e in trainer.state.log_history if "eval_f1" in e]

print(f"{'epoch':>6} {'loss':>8} {'f1':>8} {'accuracy':>9}")
for entry in history:
    print(f"{entry['epoch']:>6.0f} {entry['eval_loss']:>8.4f} "
          f"{entry['eval_f1']:>8.4f} {entry['eval_accuracy']:>9.4f}")

baseline_epochs = sum(1 for e in history if abs(e["eval_f1"] - 0.7975) < 0.001)
final_loss = history[-1]["eval_loss"]

print()
print(f"epochs stuck at the always-SCAM baseline: {baseline_epochs}  (v1 had 2)")
print(f"final eval loss: {final_loss:.4f}  (v1 ended at 0.1948, still falling)")
print(f"best f1: {max(e['eval_f1'] for e in history):.4f}")

## 6. Scores

Against the always-SCAM baseline, as before. Necessary but not sufficient — v1 passed
this section comfortably.

In [ ]:
metrics = trainer.evaluate()
prediction = trainer.predict(val_dataset)
probabilities = torch.softmax(torch.tensor(prediction.predictions), dim=-1)[:, 1].numpy()
predicted = np.argmax(prediction.predictions, axis=1)
actual = prediction.label_ids

base_precision, base_recall, base_f1, _ = precision_recall_fscore_support(
    actual, np.ones_like(actual), average="binary", zero_division=0
)

print(f"{'':<12} {'accuracy':>9} {'precision':>10} {'recall':>8} {'f1':>8}")
print(f"{'electra v2':<12} {metrics['eval_accuracy']:>9.4f} "
      f"{metrics['eval_precision']:>10.4f} {metrics['eval_recall']:>8.4f} "
      f"{metrics['eval_f1']:>8.4f}")
print(f"{'electra v1':<12} {0.9684:>9.4f} {0.9688:>10.4f} {0.9841:>8.4f} {0.9764:>8.4f}")
print(f"{'always-SCAM':<12} {accuracy_score(actual, np.ones_like(actual)):>9.4f} "
      f"{base_precision:>10.4f} {base_recall:>8.4f} {base_f1:>8.4f}")
print()
print(classification_report(
    actual, predicted, target_names=["LEGITIMATE", "SCAM"], digits=4, zero_division=0
))

matrix = confusion_matrix(actual, predicted)
print("                 predicted")
print("                 LEGIT   SCAM")
print(f"actual LEGIT     {matrix[0][0]:>5}  {matrix[0][1]:>5}   <- false alarms")
print(f"actual SCAM      {matrix[1][0]:>5}  {matrix[1][1]:>5}")
print( "                     ^ missed scams")

## 7. The checks v1 failed

This is the section that matters. Three tests, each printing PASS or FAIL.

**Confidence spread.** A model that has learned a boundary produces confident
probabilities. v1 put 0% of validation rows outside the 0.1–0.9 band; XLM-R put 100%
outside it. Anything under ~60% here means the model is still hedging around 0.5.

**Held-out legitimate messages.** Hand-written, never in any split, English and Malay,
short lines and a full transcript. v1 called every one of them SCAM. These are trivial
for a human and they are the closest thing here to a real deployment input.

**Real conversations.** The handful of non-synthetic rows in the validation set. Small,
but the only genuine data in the whole pipeline.

In [ ]:
# trainer.model, not the bare `model` variable: load_best_model_at_end restores the
# best checkpoint through the trainer, so this guarantees the probes below score the
# same weights that produced the F1 in section 6.
best_model = trainer.model
best_model.eval()


def scam_probability(text):
    with torch.no_grad():
        encoded = tokenizer(
            text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(best_model.device)
        return float(best_model(**encoded).logits.softmax(-1)[0][1])


print("=" * 68)
print("CHECK 1 - CONFIDENCE SPREAD")
print("=" * 68)
confident = ((probabilities < 0.1) | (probabilities > 0.9)).mean()
print(f"  p(scam) < 0.1 : {(probabilities < 0.1).sum():>3}")
print(f"  0.1 - 0.5     : {((probabilities >= 0.1) & (probabilities < 0.5)).sum():>3}")
print(f"  0.5 - 0.9     : {((probabilities >= 0.5) & (probabilities <= 0.9)).sum():>3}")
print(f"  p(scam) > 0.9 : {(probabilities > 0.9).sum():>3}")
print(f"  confident share: {confident:.1%}   (v1: 0.0%, XLM-R: 100%)")
check_1 = confident >= 0.6
print(f"  {'PASS' if check_1 else 'FAIL'}")

print()
print("=" * 68)
print("CHECK 2 - HAND-WRITTEN MESSAGES THE MODEL HAS NEVER SEEN")
print("=" * 68)
FRAGMENTS = [
    ("LEGITIMATE", "Sure, you can view the unit this Saturday."),
    ("LEGITIMATE", "The rent is RM1200 a month, deposit is two months as usual."),
    ("LEGITIMATE", "Boleh datang tengok rumah hujung minggu ni, saya ada."),
    ("LEGITIMATE", "Sewa rm900 sebulan, deposit 2 bulan. Agreement kita buat kat office."),
    ("SCAM", "Transfer the deposit first, then I will post you the keys."),
    ("SCAM", "Bayar deposit dulu baru boleh viewing, saya outstation sekarang."),
]

TRANSCRIPTS = [
    ("LEGITIMATE",
     "Renter: Hi, saw your ad for the condo in Subang, still available?\n"
     "Owner: Hi, yes it is. May I know when you plan to move in?\n"
     "Renter: Early next month. Can I view it first?\n"
     "Owner: Sure, come this Saturday afternoon, I'll be there to open up.\n"
     "Renter: How much is the rent and deposit?\n"
     "Owner: RM1200 monthly, two months deposit. We sign the tenancy agreement "
     "after you view - no payment before that."),
    ("SCAM",
     "Renter: Is the room still available?\n"
     "Owner: Yes. Boleh pergi viewing dulu tak?\n"
     "Owner: I'm outstation in Sabah, cannot open the unit. Kalau nak lock the room, "
     "transfer deposit RM800 dulu. Ramai orang tengah tanya."),
]


def probe(cases):
    missed = 0
    for truth, text in cases:
        probability = scam_probability(text)
        verdict = "SCAM" if probability >= 0.5 else "LEGITIMATE"
        ok = verdict == truth
        missed += not ok
        print(f"  {truth:<11} p(scam) {probability:.4f} -> {verdict:<11}"
              f"{'' if ok else '   <-- WRONG'}")
        print(f"    {text[:72].replace(chr(10), ' ')}...")
    return missed


# Scored separately: the model is trained only on multi-turn transcripts, so a bare
# one-line message is out of distribution in FORMAT as well as content. Mixing the
# two into one counter cannot tell "the model is broken" apart from "the model needs
# a whole conversation", which are different problems with different fixes.
print("-- full transcripts (the trained input format) --")
transcript_wrong = probe(TRANSCRIPTS)
print()
print("-- short fragments (out of distribution by length) --")
fragment_wrong = probe(FRAGMENTS)

wrong = transcript_wrong + fragment_wrong
check_2 = transcript_wrong == 0
print()
print(f"  transcripts: {transcript_wrong} of {len(TRANSCRIPTS)} wrong   <- this gates PASS")
print(f"  fragments:   {fragment_wrong} of {len(FRAGMENTS)} wrong   "
      f"<- advisory, see note below")
print(f"  {'PASS' if check_2 else 'FAIL'}")
if transcript_wrong == 0 and fragment_wrong > 0:
    print("  Conversation classifier works; short fragments are unreliable. The OCR")
    print("  path feeds screenshot-sized chunks, so treat short input with caution.")

print()
print("=" * 68)
print("CHECK 3 - REAL (NON-SYNTHETIC) VALIDATION CONVERSATIONS")
print("=" * 68)
if "origin" in val_frame.columns:
    real = val_frame[val_frame["origin"] == "real"]
    real_wrong = 0
    for _, row in real.iterrows():
        probability = scam_probability(row["text"])
        verdict = "SCAM" if probability >= 0.5 else "LEGITIMATE"
        ok = verdict == row["label"]
        real_wrong += not ok
        print(f"  actual {row['label']:<11} p(scam) {probability:.4f} -> {verdict:<11}"
              f"{'' if ok else '   <-- WRONG'}")
    check_3 = real_wrong == 0
    print(f"  {real_wrong} of {len(real)} wrong   (v1 got the legitimate one wrong)")
    print(f"  {'PASS' if check_3 else 'FAIL'}")
else:
    check_3 = None
    print("  no origin column - skipped")

print()
print("=" * 68)
verdict_checks = [c for c in (check_1, check_2, check_3) if c is not None]
if all(verdict_checks):
    print("ALL CHECKS PASSED - this model is worth deploying and comparing.")
else:
    print("NOT READY - see which check failed:")
    if not check_1:
        print("  Confidence spread FAILED: still has not converged. This is v1's")
        print("  failure. Raise the learning rate to 1e-4 and re-run.")
    if not check_2:
        print("  Transcripts FAILED: wrong on full conversations, the format it was")
        print("  trained on. Check the label mapping and the data before tuning.")
    if check_3 is False:
        print("  Real conversations FAILED: fits synthetic data but not real text.")
        print("  More real conversations is the fix, not more hyperparameters.")
print("=" * 68)

## 8. Save and download

In [ ]:
import json
import shutil

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

report = {
    "model": MODEL,
    "version": "v2",
    "split_set": SPLIT_SET,
    "learning_rate": LEARNING_RATE,
    "epochs_configured": EPOCHS,
    "epochs_run": len(history),
    "max_length": MAX_LENGTH,
    "class_weights": USE_CLASS_WEIGHTS,
    "accuracy": round(float(metrics["eval_accuracy"]), 4),
    "precision": round(float(metrics["eval_precision"]), 4),
    "recall": round(float(metrics["eval_recall"]), 4),
    "f1": round(float(metrics["eval_f1"]), 4),
    "confident_share": round(float(confident), 4),
    "held_out_wrong": int(wrong),
    "checks": {
        "confidence_spread": bool(check_1),
        "held_out_messages": bool(check_2),
        "real_conversations": None if check_3 is None else bool(check_3),
    },
    "confusion_matrix": {
        "legit_correct": int(matrix[0][0]),
        "false_alarms": int(matrix[0][1]),
        "missed_scams": int(matrix[1][0]),
        "scam_correct": int(matrix[1][1]),
    },
    "final_eval_loss": round(float(history[-1]["eval_loss"]), 4),
    "epoch_history": [
        {k: round(float(v), 4) for k, v in e.items()
         if k in ("epoch", "eval_loss", "eval_f1", "eval_accuracy")}
        for e in history
    ],
}

with open(f"electra-v2-results-{SPLIT_SET}.json", "w") as handle:
    json.dump(report, handle, indent=2)

print(json.dumps(report, indent=2))

archive = shutil.make_archive(FINAL_DIR, "zip", FINAL_DIR)
print(f"\n{os.path.getsize(archive) / 1e6:.0f} MB")
files.download(archive)
files.download(f"electra-v2-results-{SPLIT_SET}.json")

## 9. Afterwards

Send me the JSON from section 8 — it carries the three check results, not just F1, so
I can tell straight away whether this is deployable.

**If all checks passed:** unzip to `nlp-service/models/electra-malay-scam-random-v2`
and change one line in `nlp-service/.env`:

```
FINETUNED_MODEL_DIR=./models/electra-malay-scam-random-v2
```

`PRIMARY_MODEL` already points at the right base checkpoint. Delete the WARNING
comment block in `.env` once v2 is in — it describes v1's behaviour.

**If it failed:** leave `.env` alone and tell me which check failed and what the epoch
curve looked like. The remedy differs depending on whether it failed to converge again
or converged and still over-flags.

Keep `models/electra-malay-scam-random` (v1) and `models/xlm-roberta-scam-random`
either way. v1 is the evidence for the "high F1, useless model" section of your
report, which is a more interesting finding than any of the scores.